In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
#getting Dataset
data = pd.read_csv('data/mushrooms.csv')

In [3]:
shape = data.shape
print(f"Dataset has {shape[0]} rows and {shape[1]} columns")


Dataset has 8124 rows and 23 columns


In [4]:
data.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [5]:
le= LabelEncoder()
data_encoder = data.apply(le.fit_transform)

In [6]:
data_encoder

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,1,5,2,4,1,6,1,0,1,4,...,2,7,7,0,2,1,4,2,3,5
1,0,5,2,9,1,0,1,0,0,4,...,2,7,7,0,2,1,4,3,2,1
2,0,0,2,8,1,3,1,0,0,5,...,2,7,7,0,2,1,4,3,2,3
3,1,5,3,8,1,6,1,0,1,5,...,2,7,7,0,2,1,4,2,3,5
4,0,5,2,3,0,5,1,1,0,4,...,2,7,7,0,2,1,0,3,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,0,3,2,4,0,5,0,0,0,11,...,2,5,5,0,1,1,4,0,1,2
8120,0,5,2,4,0,5,0,0,0,11,...,2,5,5,0,0,1,4,0,4,2
8121,0,2,2,4,0,5,0,0,0,5,...,2,5,5,0,1,1,4,0,1,2
8122,1,3,3,4,0,8,1,0,1,0,...,1,7,7,0,2,1,0,7,4,2


In [7]:
data= data_encoder.values

In [8]:
x= data[:,1:]
y= data[:,0]

In [9]:
x

array([[5, 2, 4, ..., 2, 3, 5],
       [5, 2, 9, ..., 3, 2, 1],
       [0, 2, 8, ..., 3, 2, 3],
       ...,
       [2, 2, 4, ..., 0, 1, 2],
       [3, 3, 4, ..., 7, 4, 2],
       [5, 2, 4, ..., 4, 1, 2]], shape=(8124, 22))

In [10]:
y

array([1, 0, 0, ..., 0, 1, 0], shape=(8124,))

In [11]:
x_train, x_test, y_train, y_test = train_test_split(
    x,y,test_size=0.2, random_state=42
    )

<h4>Naive baised classifier</h4>

<h3> conditional probablity and prior probablity </h3>
<p>1. Conditional Probability

The formula for conditional probability is:

𝑃
(
𝐴
∣
𝐵
)
=
𝑃
(
𝐴
∩
𝐵
)
𝑃
(
𝐵
)
(where 
𝑃
(
𝐵
)
>
0
)
P(A∣B)=
P(B)
P(A∩B)
	​

(where P(B)>0)
What it means:
𝑃
(
𝐴
∣
𝐵
)
P(A∣B): Probability of A happening given that B has already happened
𝑃
(
𝐴
∩
𝐵
)
P(A∩B): Probability that both A and B happen
𝑃
(
𝐵
)
P(B): Probability that B happens

👉 In plain words:
You restrict your sample space to B, then check how often A also happens.</p>
<br>
<p>2. Prior Probability

Prior probability is simply the probability of an event before any new information is considered.

𝑃
(
𝐴
)
P(A)
What it means:
It’s the initial probability of event A
Used especially in Bayes' Theorem
Bonus: Bayes’ Theorem (connects both)
𝑃
(
𝐴
∣
𝐵
)
=
𝑃
(
𝐵
∣
𝐴
)
 
𝑃
(
𝐴
)
𝑃
(
𝐵
)
P(A∣B)=
P(B)
P(B∣A)P(A)
	​

𝑃
(
𝐴
)
P(A): prior probability
𝑃
(
𝐴
∣
𝐵
)
P(A∣B): posterior probability (updated after evidence)</p>


In [12]:
'''
Postirior Probablity = (Likelihood * Prior Probability) / Evidence
'''
def prior_prob(y_train, labels):
    m= y_train.shape[0]
    s= np.sum(y_train==labels)

    return m/s


def cond_prob(x_train, y_train, labels, feature_col, feature_val):
    x_filtered = x_train[y_train==labels]
    denominator= x_filtered.shape[0]
    s= np.sum(x_filtered[:,feature_col]==feature_val)

    return float(s/denominator)




In [13]:
def predict(x_test, x_train, y_train, labels):

    classes= np.unique(y_train)
    n_features= x_train.shape[1]
    posterior_prob= []

    for labels in classes:
        likelihood= 1.0
        for feat in range(n_features):
            cond = cond_prob(x_train, y_train, labels, feat, x_test[feat])  
            likelihood = likelihood * cond
        prior = prior_prob(y_train, labels)
        post = likelihood * prior
        posterior_prob.append(post)
        predicted_label= classes[np.argmax(posterior_prob)]
    return predicted_label

In [14]:
def accuracy(x_train, x_test, y_train, y_test):
    n_samples= x_test.shape[0]
    n_correct= 0

    for i in range(n_samples):
        predicted_label= predict(x_test[i], x_train, y_train, np.unique(y_train))
        if predicted_label == y_test[i]:
            n_correct += 1
    acc= n_correct/n_samples
    return acc

In [15]:
accuracy(x_train, x_test, y_train, y_test)*100

99.63076923076923